# memopro CUDA check, run 4: access layer and γ (docs/research/0053)

Checks on a real NVIDIA GPU the conditions pre-registered for v0.2 (0052 E2, amended in 0053):

| section | condition |
|---|---|
| `check_accuracy` (A2-3) | `memopro.check` predicts peak memory within ±15% of CUDA `max_memory_allocated` |
| `train_capped` (A2-2) | under a device cap where a plain step runs out of memory, `train_session` trains; SGD parameters and gradients match the uncapped full-batch run (relative ≤ 1e-5); AdamW is reported |
| `baseline_batch_finder` (A2-4) | the same capped task with accelerate's `find_executable_batch_size` (reported, no pass/fail) |
| `load_small` | `memopro.load` on CUDA picks half precision / int8 / int4 as the budget shrinks and generates |
| `activation_offload` | `train_session`'s activation offload step is exact (SGD, ≤ 1e-5) |
| `pressure` | the Linux memory-pressure signal (PSI) is readable (recorded) |
| `load_7b` (A2-1b) | a 7B model whose fp16 weights exceed the T4 budget loads with int8/int4 and generates |

1. **Runtime -> Change runtime type -> T4 GPU**.
2. Upload the wheel sent with run 4 into `/content` (file panel).
3. **Runtime -> Run all** (restart the session first if memopro was installed before).
   `load_7b` downloads about 15 GB; set `RUN_7B = False` in the first cell to skip it.
4. Copy the JSON printed by the last cell back to the conversation.

In [ ]:
import glob, json, os, subprocess, sys, time, traceback

RUN_7B = True
wheels = sorted(glob.glob("/content/memopro-*.whl"))
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    wheels[-1]], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=False)
import memopro, torch, transformers, accelerate  # noqa: E401, E402

results = {"memopro": memopro.__version__, "python": sys.version.split()[0],
           "torch": torch.__version__, "transformers": transformers.__version__,
           "accelerate": accelerate.__version__,
           "wheel": os.path.basename(wheels[-1]) if wheels else None,
           "access_0053": hasattr(memopro.access, "train_session")
           and "reduction" in memopro.access._train.TrainSession.__init__.__code__.co_varnames,
           "cuda": torch.cuda.is_available(),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
try:
    import bitsandbytes
    results["bitsandbytes"] = bitsandbytes.__version__
except Exception as e:
    results["bitsandbytes"] = f"missing: {e}"


def section(name):
    def wrap(fn):
        t = time.time()
        try:
            results[name] = fn()
        except Exception as e:  # keep going: report everything
            results[name] = {"error": f"{type(e).__name__}: {e}"[:400],
                             "trace": traceback.format_exc()[-800:]}
        print(name, f"({time.time() - t:.0f}s) ->", json.dumps(results[name], default=str)[:400])
        cleanup()
    return wrap


def cleanup():
    import gc
    for h in memopro.hibernate.handles():
        if h.asleep:
            try:
                h.wake()
            except Exception:
                h.discard()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.set_per_process_memory_fraction(1.0)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def max_rel(a, b):
    return max(float((x.float() - y.float()).norm() / y.float().norm()) for x, y in zip(a, b))

print(results)

In [ ]:
@section("pressure")
def _():
    from memopro import elastic
    try:
        return {"reading": elastic.current()}
    except memopro.ModeUnavailable as e:
        return {"unsupported": e.reason}

In [ ]:
from transformers import AutoModelForCausalLM


def measured_peak(name, goal, batch, seq):
    """Steady-state CUDA peak of the same work check predicts (fp32, AdamW for training)."""
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).cuda()
    x = torch.randint(0, model.config.vocab_size, (batch, seq), device="cuda")
    try:
        if goal == "infer":
            model.eval()
            with torch.no_grad():
                model(x)  # warm up (cuBLAS workspace, kernels)
                torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
                model(x)
        else:
            opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
            model.train()
            for step in range(2):  # the second step has optimizer state from the start
                if step == 1:
                    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
                model(x, labels=x).loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated()
    except torch.OutOfMemoryError:
        return "oom"
    finally:
        del model
        cleanup()


def accelerate_estimate(name):
    """accelerate's estimate (weights; training with Adam counted as 4x the weights)."""
    try:
        from accelerate.commands.estimate import calculate_maximum_sizes, create_empty_model
        model = create_empty_model(name, library_name="transformers", trust_remote_code=False)
        total, _ = calculate_maximum_sizes(model)
        return {"weights": int(total), "train_adam": 4 * int(total)}
    except Exception as e:
        return {"weights": None, "train_adam": None, "error": f"{type(e).__name__}: {e}"[:120]}


@section("check_accuracy")
def _():
    cases = [("openai-community/gpt2", "infer", 1, 1024), ("openai-community/gpt2", "train", 2, 512),
             ("openai-community/gpt2", "train", 8, 512),
             ("openai-community/gpt2-medium", "infer", 1, 1024),
             ("openai-community/gpt2-medium", "train", 1, 512),
             ("openai-community/gpt2-medium", "train", 2, 512)]
    out = []
    for name, goal, batch, seq in cases:
        r = memopro.check(name, goal=goal, batch_size=batch, seq_len=seq, device="cuda")
        pred = (r.inference if goal == "infer" else r.training).get("peak")
        real = measured_peak(name, goal, batch, seq)
        err = None if real == "oom" or not pred else round(pred / real - 1, 4)
        acc = accelerate_estimate(name)
        acc_value = acc["weights"] if goal == "infer" else acc["train_adam"]
        acc_err = None if real == "oom" or not acc_value else round(acc_value / real - 1, 4)
        fits = (r.training.get("fits") if goal == "train" else None)
        out.append({"model": name.split("/")[-1], "goal": goal, "batch": batch, "seq": seq,
                    "predicted": pred, "measured": real, "rel_error": err,
                    "within_15pct": err is not None and abs(err) <= 0.15,
                    "check_said_fits": fits, "accelerate": acc_value,
                    "accelerate_rel_error": acc_err})
    measured = [c for c in out if c["measured"] != "oom"]
    return {"cases": out, "all_within_15pct": all(c["within_15pct"] for c in measured),
            "measured_cases": len(measured)}

In [ ]:
def gpt2_small():
    torch.manual_seed(0)
    return AutoModelForCausalLM.from_pretrained("openai-community/gpt2", dtype=torch.float32,
                                                attn_pdrop=0.0, resid_pdrop=0.0,
                                                embd_pdrop=0.0).cuda()


BATCH = {"input_ids": torch.randint(0, 50257, (8, 256), generator=torch.Generator().manual_seed(0))}


def plain_step(model, opt):
    x = BATCH["input_ids"].cuda()
    model(x, labels=x).loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)


def capped(fraction):
    torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(fraction)


@section("train_capped")
def _():
    out = {}
    for name, make in (("sgd", lambda p: torch.optim.SGD(p, lr=1e-2)),
                       ("adamw", lambda p: torch.optim.AdamW(p, lr=1e-4))):
        ref = gpt2_small(); ropt = make(ref.parameters())
        for _ in range(2):
            plain_step(ref, ropt)
        ref_params = [p.detach().cpu() for p in ref.parameters()]
        del ref, ropt
        cleanup()
        fraction = 0.3  # about 4.4 GB of the T4: a plain step with batch 8 does not fit
        model = gpt2_small(); opt = make(model.parameters())
        capped(fraction)
        try:
            plain_step(model, opt)
            plain = "fits"
        except torch.OutOfMemoryError:
            plain = "oom"
        del model, opt
        cleanup()
        model = gpt2_small(); opt = make(model.parameters())
        capped(fraction)
        t = time.time()
        with memopro.train_session(model, opt, micro_batch_size=8) as s:
            for _ in range(2):
                s.step({"input_ids": BATCH["input_ids"].cuda()},
                       lambda mb: model(**mb, labels=mb["input_ids"]).loss)
            session = {"micro": s.micro, "retries": s.retries, "active": s.active(),
                       "seconds": round(time.time() - t, 1)}
        diff = max_rel([p.detach().cpu() for p in model.parameters()], ref_params)
        out[name] = {"plain_step_under_cap": plain, "session": session, "param_rel_diff": diff}
        del model, opt
        cleanup()
    out["pass"] = (out["sgd"]["plain_step_under_cap"] == "oom"
                   and out["sgd"]["param_rel_diff"] <= 1e-5)
    return out


@section("baseline_batch_finder")
def _():
    from accelerate.utils import find_executable_batch_size
    model = gpt2_small(); opt = torch.optim.SGD(model.parameters(), lr=1e-2)
    capped(0.3)
    tries = []
    t = time.time()

    @find_executable_batch_size(starting_batch_size=8)
    def train(batch_size):
        tries.append(batch_size)
        x = BATCH["input_ids"][:batch_size].cuda()
        model(x, labels=x).loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        return batch_size

    final = train()
    return {"tries": tries, "final_batch": final, "seconds": round(time.time() - t, 1),
            "note": "the batch shrinks: the effective batch (and the result) changes"}

In [ ]:
@section("load_small")
def _():
    name = "openai-community/gpt2"
    out = {}
    plan = memopro.access._load.plan_load(name, device="cuda", quality="low")
    need = {c.name: c.needs.device for c in plan.candidates}
    # budgets just above each configuration's need, so that one is the first that fits
    for label, budget, quality in (("stored", None, None),
                                   ("half", int(need["half"] * 1.02), None),
                                   ("quant.int8", int(need["quant.int8"] * 1.02), None),
                                   ("quant.int4", int(need["quant.int4"] * 1.02), "low")):
        memopro.report().clear()
        try:
            model, tok = memopro.load(name, device="cuda", budget=budget, quality=quality,
                                      tokenizer=True)
            ids = tok("The memory of a GPU is", return_tensors="pt").input_ids.cuda()
            text = tok.decode(model.generate(ids, max_new_tokens=8, do_sample=False,
                                             pad_token_id=tok.eos_token_id)[0])
            chosen = memopro.report().entries[-1].technique
            layers = sorted({type(m).__name__ for m in model.modules()
                             if "Linear" in type(m).__name__})
            out[label] = {"chosen": chosen, "as_expected": chosen == f"load.{label}",
                          "layers": layers, "text": text[:60],
                          "allocated": torch.cuda.memory_allocated(), "budget": budget}
            del model
        except Exception as e:
            out[label] = {"error": f"{type(e).__name__}: {e}"[:300]}
        cleanup()
    out["pass"] = all(v.get("as_expected") for v in out.values() if isinstance(v, dict))
    return out


@section("activation_offload")
def _():
    ref = gpt2_small(); ropt = torch.optim.SGD(ref.parameters(), lr=1e-2)
    plain_step(ref, ropt)
    ref_params = [p.detach().cpu() for p in ref.parameters()]
    del ref, ropt
    cleanup()
    model = gpt2_small(); opt = torch.optim.SGD(model.parameters(), lr=1e-2)
    with memopro.train_session(model, opt, micro_batch_size=2, quality="lossless") as s:
        while not s.activation_offload:
            assert s.degrade("test"), "no activation offload step"
        s.step({"input_ids": BATCH["input_ids"].cuda()},
               lambda mb: model(**mb, labels=mb["input_ids"]).loss)
        active = s.active()
    diff = max_rel([p.detach().cpu() for p in model.parameters()], ref_params)
    return {"active": active, "param_rel_diff": diff, "pass": diff <= 1e-5}

In [ ]:
@section("load_7b")
def _():
    if not RUN_7B:
        return "skipped: RUN_7B is False"
    name = "Qwen/Qwen2.5-7B-Instruct"
    plan = memopro.access._load.plan_load(name, device="cuda")
    half = plan.info.weight_bytes(half=True)
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    model, tok = memopro.load(name, device="cuda", tokenizer=True)
    load_s = time.time() - t
    msgs = [{"role": "user", "content": "In one sentence, why do GPUs run out of memory?"}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to("cuda")
    out = model.generate(**enc, max_new_tokens=40, do_sample=False)
    text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    peak = torch.cuda.max_memory_allocated()
    chosen = [e for e in memopro.report().entries if e.technique.startswith("load.")][-1]
    return {"half_weights": half, "device_budget": plan.ctx.device_budget,
            "half_fits": half + plan.ctx.runtime_bytes <= plan.ctx.device_budget,
            "chosen": chosen.technique, "detail": chosen.detail[:200], "peak": peak,
            "pass": peak <= plan.ctx.device_budget and len(text) > 0,
            "load_seconds": round(load_s), "text": text[:200]}

In [ ]:
print(json.dumps(results, indent=2, default=str))